# Independent-split certificate coverage
Generates results/coverage_indep/. Pure order-3 at pair-0.9, sigma swept to move the true gap through the resolvable floor.

In [ ]:
import numpy as np, json, csv, time, hashlib, zlib, inspect, sys, platform
import scipy
from itertools import product as iproduct
from scipy import stats
import os
OUTBASE = '/content/drive/MyDrive/ORDER_SWEEP/results'

def monomial_exps(d, D, k):
    return [c for c in iproduct(range(D+1), repeat=d)
            if sum(c) <= D and sum(1 for x in c if x > 0) <= k]
def poly_design(X, D, k):
    exps = monomial_exps(X.shape[1], D, k); cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0: col = col * X[:, j]**p
        cols.append(col)
    return np.column_stack(cols), exps.index(tuple([0]*X.shape[1]))
def select_order(X, h, K, S=10, alpha=0.05, D=4, train_frac=0.75):
    n = X.shape[0]; n_tr = int(train_frac*n); n_te = n - n_tr
    designs = {k: poly_design(X, D, k) for k in range(1, K+1)}; r2 = {}
    for s in range(S):
        idx = np.random.default_rng(s).permutation(n); tr, te = idx[:n_tr], idx[n_tr:]; out = {}
        for k in range(1, K+1):
            P0, ci = designs[k]; mu = P0[tr].mean(0); sd = P0[tr].std(0); sd[sd == 0] = 1.0
            P = (P0 - mu)/sd; P[:, ci] = 1.0; hm = h[tr].mean()
            beta, *_ = np.linalg.lstsq(P[tr], h[tr]-hm, rcond=None); resid = (h[te]-hm) - P[te]@beta
            out[k] = 1.0 - float((resid@resid)/np.sum((h[te]-h[te].mean())**2))
        r2[s] = out
    corr = 1.0/S + n_te/n_tr; pf = {k: designs[k][0].shape[1] for k in range(1, K+1)}
    om = float(np.mean([1.0 - r2[s][K] for s in range(S)])); stat = {}
    for k in range(1, K):
        g = np.array([r2[s][K] - r2[s][k] for s in range(S)]); m = g.mean(); v = g.var(ddof=1)*corr
        opt = (pf[K] - pf[k])*om/n_tr
        if v > 0:
            t = m/np.sqrt(v); p = 1.0 - stats.t.cdf(t, df=S-1); ub = m + stats.t.ppf(1-alpha, df=S-1)*np.sqrt(v) + opt
        else:
            p = 0.0 if m > 0 else 1.0; ub = m + opt
        stat[k] = {"mean": float(m), "p": float(p), "ub": float(ub)}
    khat, ubc = K, None
    for k in range(1, K):
        if stat[k]["p"] > alpha: khat, ubc = k, stat[k]["ub"]; break
    return khat, stat, ubc

# ---- independent-split coverage study ----
OUT = os.path.join(OUTBASE, "coverage_indep"); os.makedirs(OUT, exist_ok=True)
F=lambda r:(1-r**2)**2/((1+r**2)*(1+2*r**2)); VH=1+2*0.81
def truth(sig): return F(0.9)*VH/(VH+sig**2)
N,R=20_000,50; SIGMAS=[1.0,2.0,3.0,4.0,5.0,6.0]
rows=[]
for sig in SIGMAS:
    for rep in range(R):
        seed=(60_000+zlib.crc32(f"cov|{sig}".encode())+rep*977)%2**32
        rng=np.random.default_rng(seed)
        x1,x2,z=rng.standard_normal((3,N)); x3=0.9*x1+np.sqrt(0.19)*z
        X=np.column_stack([x1,x2,x3]); h=x1*x2*x3+sig*rng.standard_normal(N)
        dh=hashlib.sha256(X.tobytes()+h.tobytes()).hexdigest()
        kh,st,ub=select_order(X,h,K=3)
        rows.append({"experiment":"coverage_indep","sigma":sig,"rep":rep,"data_hash":dh,
                     "khat":kh,"rem2_mean":st[2]["mean"],"rem2_p":st[2]["p"],
                     "ub_cert":"" if ub is None else ub,"true_gap":truth(sig)})
with open(os.path.join(OUT,"per_seed_coverage.csv"),"w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print("wrote", len(rows), "rows to coverage_indep/per_seed_coverage.csv")
